# PDF Extraction

In [3]:
# Teste PDF Extraction
import sys
from pathlib import Path
def _add_src_to_path():
    from pathlib import Path
    import sys
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path
    raise RuntimeError("Konnte den Projektpfad nicht finden. Erwartet wurde ein Ordner 'src/byeias'.")
_add_src_to_path()
from byeias.backend.extraction.text_extracter import PDFTextExtractor

# Beispiel-PDF (muss existieren)
pdf_path = "../../../../data/Businessplan_Halo_AktuellerStand.pdf"

extractor = PDFTextExtractor(language="german")
# Prüfe, ob die PDF-Datei existiert, bevor extrahiert wird

pdf_file = Path(pdf_path)
if not pdf_file.exists():
    print(f"Datei nicht gefunden: {pdf_file.resolve()}")
    sentences = []
else:
    sentences = extractor.extract_sentences(pdf_path)
    
    print(f"Extrahierte Sätze: {len(sentences)}")
for i, satz in enumerate(sentences[:5], 1):
    print(f"{i}: {satz}")

Extrahierte Sätze: 346
1: Business Plan Halo Navigator Projektbericht an der Fakult¨at Wirtschaft im Studiengang Data Science und K ¨unstliche Intelligenz an der DHBW Ravensburg Autoren/Autorinnen: David Muth, Nischal Parajuli, Jannes Kurzke Modul: Entrepreneurship - Business Planning Dozent: Marcus Bentele Abgabefrist: 31.03.2026 Inhaltsverzeichnis Inhaltsverzeichnis Abk¨urzungsverzeichnis III Abbildungsverzeichnis IV 1 Executive Summary Jannes 1 2 Produkt/Dienstleistung Nischal 2 3 Markt und Wettbewerb Nischal 3 4 Marketing und Vertrieb Nischal 4 5 Gesch ¨aftsmodell und Organisation David 5 6 Team David 6 7 Realisierungsfahrplan - Jannes 7 8 Chancen und Risiken - Jannes 8 9 Finanzplanung und Finanzierung 9 9.1 Annahmen .
2: .
3: .
4: .
5: .


# Classification

In [4]:
import logging
from pathlib import Path

import pandas as pd

def _add_src_to_path():
    """Findet den Projektroot und fügt <root>/src zu sys.path hinzu."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path

    raise RuntimeError(
        "Konnte den Projektpfad nicht finden. Erwartet wurde ein Ordner 'src/byeias'."
    )

SRC_PATH = _add_src_to_path()
print(f"Nutze Python-Pfad: {SRC_PATH}")

# Importiere BiasDetectionPipeline aus dem richtigen Modulpfad
from byeias.backend.classification.model_bias import BiasDetectionPipeline  # noqa: E402

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

Nutze Python-Pfad: C:\Users\david\OneDrive\Desktop\Byeias\src


c:\Users\david\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def build_tiny_datasets():
    """Sehr kleine Datensätze, um die Pipeline-Funktionalität schnell zu testen."""
    train_df_sexism = pd.DataFrame(
        {
            "context": [
                "Im Teammeeting ging es um Führungsrollen.",
                "Im Büro wurde über neue Projekte gesprochen.",
                "Bei der Bewerbung ging es um Qualifikationen.",
                "In der Schule wurden die Noten verteilt.",
            ],
            "text": [
                "Frauen sind zu emotional für Leitungspositionen.",
                "Alle haben konstruktiv zusammengearbeitet.",
                "Männer sind besser für Technikjobs geeignet.",
                "Die Klasse hat fair zusammen gelernt.",
            ],
            "sexism_label": [1, 0, 1, 0],
        }
    )

    train_df_racism = pd.DataFrame(
        {
            "context": [
                "In einer Diskussion über Migration.",
                "Im Stadtpark fand ein Fest statt.",
                "Im Unterricht ging es um kulturelle Vielfalt.",
                "Beim Sportevent waren viele Nationen vertreten.",
            ],
            "text": [
                "Menschen aus diesem Land sind alle kriminell.",
                "Alle Familien haben zusammen gefeiert.",
                "Ausländer wollen sich nie integrieren.",
                "Das Turnier lief respektvoll und fair.",
            ],
            "racism_label": [1, 0, 1, 0],
        }
    )

    val_df_sexism = pd.DataFrame(
        {
            "context": [
                "Im Bewerbungsgespräch wurde Leistung bewertet.",
                "Im Verein planten alle gemeinsam das Event.",
            ],
            "text": [
                "Frauen sollten lieber nicht in die Chefetage.",
                "Die Aufgaben wurden unabhängig vom Geschlecht verteilt.",
            ],
            "sexism_label": [1, 0],
        }
    )

    val_df_racism = pd.DataFrame(
        {
            "context": [
                "In der Kantine wurde über Herkunft gesprochen.",
                "Im Seminar wurden Beispiele aus vielen Ländern gezeigt.",
            ],
            "text": [
                "Diese Ethnie ist grundsätzlich faul.",
                "Unterschiedliche Hintergründe bereichern die Diskussion.",
            ],
            "racism_label": [1, 0],
        }
    )

    return train_df_sexism, train_df_racism, val_df_sexism, val_df_racism

In [6]:
def build_test_samples(n=5):
    """Erzeugt n Testbeispiele für die Vorhersage."""
    samples = [
        {
            "context": "Im Klassenzimmer ging es um Mathematik.",
            "text": "Mädchen sind von Natur aus schlechter in Mathe.",
        },
        {
            "context": "Im Unternehmen wurde ein neues Team gebildet.",
            "text": "Die Zusammenarbeit lief professionell und respektvoll.",
        },
        {
            "context": "In einer Debatte über Einwanderung.",
            "text": "Alle Migranten sind eine Belastung.",
        },
        {
            "context": "Beim Stadtfest waren viele Kulturen vertreten.",
            "text": "Das Fest war offen und inklusiv.",
        },
        {
            "context": "Im Projektmeeting wurden Rollen verteilt.",
            "text": "Frauen sollten lieber unterstützende Aufgaben übernehmen.",
        },
    ]
    return samples[:n]

In [7]:
n_predictions = 5
train_df_sexism, train_df_racism, val_df_sexism, val_df_racism = build_tiny_datasets()
test_samples = build_test_samples(n=n_predictions)
test_contexts = [sample["context"] for sample in test_samples]
test_texts = [sample["text"] for sample in test_samples]

print(f"Train Sexism: {len(train_df_sexism)} | Train Racism: {len(train_df_racism)}")
print(f"Val Sexism: {len(val_df_sexism)} | Val Racism: {len(val_df_racism)}")
print(f"Inference Beispiele: {len(test_texts)}")

Train Sexism: 4 | Train Racism: 4
Val Sexism: 2 | Val Racism: 2
Inference Beispiele: 5


In [8]:
logger.info("Initialisiere Pipeline...")
pipeline = BiasDetectionPipeline(model_name="microsoft/deberta-v3-small")
print("Pipeline initialisiert")

INFO:__main__:Initialisiere Pipeline...
c:\Users\david\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Pipeline initialisiert


In [9]:
logger.info("Starte kurzes Training...")
pipeline.train(
    train_df_sexism=train_df_sexism,
    train_df_racism=train_df_racism,
    val_df_sexism=val_df_sexism,
    val_df_racism=val_df_racism,
    epochs=1,
    batch_size=2,
    lr=2e-5,
    )
print("Training abgeschlossen")

INFO:__main__:Starte kurzes Training...
Epoch 1/1 [Train]: 100%|██████████| 4/4 [00:09<00:00,  2.30s/it, loss=0.6855]


Training abgeschlossen


In [10]:
logger.info("Teste Inference auf n Beispielen...")
predictions = pipeline.predict(context_texts=test_contexts, target_texts=test_texts)
print(f"Vorhersagen erstellt: {len(predictions)}")

INFO:__main__:Teste Inference auf n Beispielen...


Vorhersagen erstellt: 5


In [11]:
print("\n--- INFERENCE ERGEBNISSE ---")
for idx, res in enumerate(predictions, start=1):
    print(f"Beispiel {idx}:")
    print(f"Kontext: '{res['context']}'")
    print(f"Text:    '{res['text']}'")
    print(f" -> Sexismus: {'JA' if res['sexism_prediction'] == 1 else 'NEIN'}")
    print(f" -> Rassismus: {'JA' if res['racism_prediction'] == 1 else 'NEIN'}\n")


--- INFERENCE ERGEBNISSE ---
Beispiel 1:
Kontext: 'Im Klassenzimmer ging es um Mathematik.'
Text:    'Mädchen sind von Natur aus schlechter in Mathe.'
 -> Sexismus: JA
 -> Rassismus: NEIN

Beispiel 2:
Kontext: 'Im Unternehmen wurde ein neues Team gebildet.'
Text:    'Die Zusammenarbeit lief professionell und respektvoll.'
 -> Sexismus: JA
 -> Rassismus: NEIN

Beispiel 3:
Kontext: 'In einer Debatte über Einwanderung.'
Text:    'Alle Migranten sind eine Belastung.'
 -> Sexismus: JA
 -> Rassismus: NEIN

Beispiel 4:
Kontext: 'Beim Stadtfest waren viele Kulturen vertreten.'
Text:    'Das Fest war offen und inklusiv.'
 -> Sexismus: JA
 -> Rassismus: NEIN

Beispiel 5:
Kontext: 'Im Projektmeeting wurden Rollen verteilt.'
Text:    'Frauen sollten lieber unterstützende Aufgaben übernehmen.'
 -> Sexismus: JA
 -> Rassismus: NEIN



# LLM Communication

In [ ]:
# Teste LLM Communication
from pathlib import Path

def _add_src_to_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        src_path = candidate / "src"
        package_path = src_path / "byeias"
        if package_path.exists():
            src_str = str(src_path)
            if src_str not in sys.path:
                sys.path.insert(0, src_str)
            return src_path
    raise RuntimeError("Konnte den Projektpfad nicht finden. Erwartet wurde ein Ordner 'src/byeias'.")

_add_src_to_path()
from byeias.backend.llm_explanation.llm_communicator import LLMCommunicator

# Beispiel-Kontext
context_before = "Im Klassenzimmer ging es um Mathematik."
flagged_sentence = "Mädchen sind von Natur aus schlechter in Mathe."
context_after = "Die Lehrerin erklärte die nächste Aufgabe."

llm = LLMCommunicator()
result = llm.explain_bias(context_before, flagged_sentence, context_after)
print("LLM-Ergebnis:")
print(result)

ImportError: attempted relative import with no known parent package